# Aula 3 — FASTA, FASTQ e SRA

Dataset principal: **PRJNA760946**  
Run: **SRR15736591** — *Hypochilus petrunkevitchi*

Esta aula inicia o pipeline principal que será continuado nas aulas seguintes.

## 0. Preparar o runtime

O Google Drive guarda os **dados** entre as aulas, mas o runtime do Colab é temporário.
Programas instalados no runtime podem desaparecer quando a sessão termina.

Por isso, quando uma aula precisar de ferramentas externas, começaremos verificando se
o Conda já existe. Se não existir, ele será instalado antes de qualquer outra configuração.

> Esta deve ser a primeira célula executável do notebook, porque a instalação do Conda
> pode reiniciar o runtime.

In [ ]:
import shutil

if shutil.which("conda"):
    print("Conda já está disponível neste runtime.")
else:
    !pip install -q condacolab
    import condacolab
    condacolab.install()

### Verificar o Conda e configurar Bioconda

Usaremos a configuração recomendada pelo Bioconda: `conda-forge` com maior prioridade,
seguido de `bioconda`, e prioridade estrita.

Como `conda config --add` adiciona canais do menor para o maior nível de prioridade,
executamos primeiro `bioconda` e depois `conda-forge`.

In [ ]:
!conda --version
!conda config --remove-key channels 2>/dev/null || true
!conda config --add channels bioconda
!conda config --add channels conda-forge
!conda config --set channel_priority strict
!conda config --show channels

## 1. Retomar o projeto no Google Drive

Todas as práticas usam a mesma raiz:

`/content/drive/MyDrive/Bioinformatica_Biologia_Molecular`

Os resultados de uma aula são lidos pela aula seguinte. Assim, os **dados persistem**
mesmo quando o runtime do Colab é encerrado.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

ROOT = Path("/content/drive/MyDrive/Bioinformatica_Biologia_Molecular")
RUN = "SRR15736591"
SAMPLE = "hypochilus_petrunkevitchi_SRR15736591"

PASTAS = {
    "01_bancos": ROOT / "01_bancos",
    "02_blast": ROOT / "02_blast",
    "03_raw": ROOT / "03_sra_fastq" / "raw-fastq",
    "04_qc": ROOT / "04_qc_trimming",
    "04_trimmed": ROOT / "04_qc_trimming" / "trimmed",
    "05_assemblies": ROOT / "05_spades" / "spades-assemblies",
    "05_contigs": ROOT / "05_spades" / "spades-assemblies" / "contigs",
    "06_match": ROOT / "06_uce_match",
    "06_probes": ROOT / "06_uce_match" / "probes",
    "06_results": ROOT / "06_uce_match" / "uce-search-results",
    "07_taxon_sets": ROOT / "07_uce_extract" / "taxon-sets" / "all",
    "08_integracao": ROOT / "08_integracao",
    "ambientes": ROOT / "ambientes",
}

for pasta in PASTAS.values():
    pasta.mkdir(parents=True, exist_ok=True)

os.chdir(ROOT)

print("Diretório atual:", Path.cwd())
print("\nEstrutura principal do projeto:")
for chave, pasta in PASTAS.items():
    print(f"{chave:15s} -> {pasta.relative_to(ROOT)}")

In [ ]:
print("\nPastas existentes na raiz:")
for p in sorted(ROOT.iterdir()):
    if p.is_dir():
        print(" -", p.name)

## 2. Localizar a etapa atual

A Aula 3 não precisa do resultado do BLAST para funcionar. Ela representa a transição
do exemplo de uma única sequência para os dados brutos de sequenciamento.

Os reads serão armazenados em:

`03_sra_fastq/raw-fastq/`

In [ ]:
OUT = PASTAS["03_raw"]
print("Diretório de saída:", OUT)

blast_anterior = PASTAS["02_blast"] / "blastn_nt.tsv"
print("Resultado da Aula 2:", "encontrado" if blast_anterior.exists() else "não encontrado (não bloqueia esta aula)")

## 3. Preparar o ambiente `bioinfo` e instalar SRA Toolkit

In [ ]:
!conda env list | grep -qE '^bioinfo[[:space:]]' || conda create -y -n bioinfo python=3.11
!conda install -y -n bioinfo sra-tools
!conda run -n bioinfo fasterq-dump --version

## 4. Definir o tamanho da amostra didática

In [ ]:
MAX_SPOTS = 200_000
print("Run:", RUN)
print("Máximo de spots:", f"{MAX_SPOTS:,}")

## 5. Obter os reads paired-end

`--split-files` gera um arquivo para cada extremidade do fragmento.

In [ ]:
outdir = str(OUT)
!conda run -n bioinfo fasterq-dump "$RUN" -X "$MAX_SPOTS" --split-files -e 2 -O "$outdir"

## 6. Compactar e conferir os arquivos

In [ ]:
!gzip -f "{OUT}/{RUN}_1.fastq"
!gzip -f "{OUT}/{RUN}_2.fastq"
!ls -lh "{OUT}"

## 7. Visualizar registros FASTQ

In [ ]:
!zcat "{OUT}/{RUN}_1.fastq.gz" | head -8

## 8. Contar reads e verificar o pareamento

In [ ]:
import gzip

R1 = OUT / f"{RUN}_1.fastq.gz"
R2 = OUT / f"{RUN}_2.fastq.gz"

def contar_reads(path):
    with gzip.open(path, "rt") as f:
        return sum(1 for _ in f) // 4

def primeiro_id(path):
    with gzip.open(path, "rt") as f:
        return f.readline().strip()

print("R1:", contar_reads(R1), primeiro_id(R1))
print("R2:", contar_reads(R2), primeiro_id(R2))

## 9. Registrar o ambiente

In [ ]:
ENV_FILE = PASTAS["ambientes"] / "aula03_bioinfo.yml"
!conda env export -n bioinfo --from-history > "$ENV_FILE"
print(ENV_FILE)

## Saída para a próxima aula

A Aula 4 verificará automaticamente:

- `03_sra_fastq/raw-fastq/SRR15736591_1.fastq.gz`
- `03_sra_fastq/raw-fastq/SRR15736591_2.fastq.gz`

Se esses arquivos não existirem, a Aula 4 interromperá a execução e indicará o que está faltando.